# Freigh Rate Prediction

In [ ]:
#Import dependencies
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

from sklearn.ensemble import RandomForestRegressor

from sklearn.ensemble import GradientBoostingRegressor




In [165]:
#Load data then explore rows and columns
df = pd.read_csv("train-test.csv")
df.shape

(48000, 14)

## EDA

In [166]:
# Check data head
df.head()

,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,0.97623,2.43355,679.97
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,1.00971,1.84491,1802.54
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,2025-01-01,0.94518,1.87712,1827.28
4,TR-000005,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,2025-01-01,0.98480,2.56300,1380.28


In [167]:
df.isnull().sum()

load_id           0
pickup            0
delivery          0
pickup_lat        0
pickup_lon        0
delivery_lat      0
delivery_lon      0
distance          0
equipment         0
weight          300
date              0
market_index    374
quote_signal      0
posted_rate       0
dtype: int64

In [168]:
df.duplicated().sum()

np.int64(0)

In [169]:
df[["weight", "market_index"]].describe()

,weight,market_index
count,47700.000000,47626.000000
mean,31028.844004,1.083387
std,9391.440620,0.168091
min,-47500.000000,0.676390
25%,25800.000000,0.949670
50%,31436.500000,1.055800
75%,37018.000000,1.219590
max,47500.000000,1.467780


In [170]:
# Noticed data quality issue (weight has negative values)
(df["weight"] < 0).sum()

np.int64(292)

In [171]:
df.loc[df["weight"] < 0, "weight"].describe()

count      292.000000
mean    -31724.195205
std       8262.536326
min     -47500.000000
25%     -37284.250000
50%     -31821.500000
75%     -25928.500000
max      -5000.000000
Name: weight, dtype: float64

In [172]:
df.loc[
    df["weight"] < 0,
    ["load_id", "weight", "distance", "equipment", "posted_rate"]
].head(20)

,load_id,weight,distance,equipment,posted_rate
68,TR-000069,-36559.0,1334.2,Reefer,2831.27
204,TR-000205,-26670.0,496.2,Dry Van,1046.92
206,TR-000207,-20003.0,411.3,Reefer,959.15
225,TR-000226,-23981.0,1317.4,Reefer,2836.00
376,TR-000377,-41397.0,1331.1,Dry Van,2522.54
446,TR-000447,-37215.0,2438.3,Reefer,4841.57
495,TR-000496,-31214.0,715.9,Dry Van,1473.78
641,TR-000642,-41023.0,582.4,Dry Van,1264.33
806,TR-000807,-28320.0,676.1,Dry Van,1307.76
1093,TR-001094,-25153.0,1058.5,Dry Van,1907.83


In [173]:
df[df["weight"] < 0][["load_id", "weight", "distance", "posted_rate"]]

,load_id,weight,distance,posted_rate
68,TR-000069,-36559.0,1334.2,2831.27
204,TR-000205,-26670.0,496.2,1046.92
206,TR-000207,-20003.0,411.3,959.15
225,TR-000226,-23981.0,1317.4,2836.00
376,TR-000377,-41397.0,1331.1,2522.54
...,...,...,...,...
47404,TR-047405,-24142.0,468.7,1077.64
47431,TR-047432,-35617.0,1978.6,4295.45
47735,TR-047736,-43951.0,1689.1,3933.12
47850,TR-047851,-28144.0,538.3,1191.61


In [174]:
df[df["weight"] < 0]["weight"].abs().describe()

count      292.000000
mean     31724.195205
std       8262.536326
min       5000.000000
25%      25928.500000
50%      31821.500000
75%      37284.250000
max      47500.000000
Name: weight, dtype: float64

In [175]:
df[df["weight"] > 0]["weight"].describe()

count    47408.000000
mean     31415.358674
std       7994.902979
min       5000.000000
25%      25922.000000
50%      31493.500000
75%      37063.000000
max      47500.000000
Name: weight, dtype: float64

Noticed:For this specific dataset, negative weights are physically invalid, and their magnitudes closely match the distribution of valid positive weights, which strongly suggests a sign error.

In [176]:
# Fix sign-error bug: negative weights are physically invalid and their magnitudes
# match the positive distribution, so take the absolute value.
df["weight"] = df["weight"].abs()

(df["weight"] < 0).sum()

np.int64(0)

In [177]:
# Check Categorical Data

In [178]:
df["equipment"].value_counts()

equipment
Dry Van    27202
Reefer     12045
Flatbed     8753
Name: count, dtype: int64

Notes:
- The dataset contains three equipment categories, with Dry Van being the most common
- Good for usage as categorical feature

In [179]:
#Inspect Dates

In [180]:

df["date"].head()

0    2025-01-01
1    2025-01-01
2    2025-01-01
3    2025-01-01
4    2025-01-01
Name: date, dtype: object

In [181]:
df["date"].dtype

dtype('O')

* Dates should be converted to datetime format

In [182]:
df[["pickup", "delivery"]].nunique()

pickup      64
delivery    64
dtype: int64

In [183]:
df[["distance", "quote_signal", "market_index"]].describe()

,distance,quote_signal,market_index
count,48000.000000,48000.000000,47626.000000
mean,1135.856654,2.062468,1.083387
std,728.564416,0.291391,0.168091
min,70.000000,0.692280,0.676390
25%,550.400000,1.891030,0.949670
50%,953.300000,2.055750,1.055800
75%,1645.525000,2.221685,1.219590
max,3439.800000,3.610350,1.467780


* market_index has missing values we may have to do imputation rather than deleting their rows

In [184]:
#Check clrrlation between target9posted_rate) and all numeric values
df.corr(numeric_only=True)["posted_rate"].sort_values(ascending=False)

posted_rate     1.000000
distance        0.908519
weight          0.041343
market_index    0.034165
quote_signal   -0.039858
pickup_lat     -0.090873
delivery_lat   -0.091970
pickup_lon     -0.255058
delivery_lon   -0.257086
Name: posted_rate, dtype: float64

* Strong correlation between distance and posted rate

In [185]:
#Check if the rate change over time?
df["date"] = pd.to_datetime(df["date"])

df.groupby(df["date"].dt.month)["posted_rate"].mean()

date
1     2255.967048
2     2273.804801
3     2372.268092
4     2372.162308
5     2421.776342
6     2497.030115
7     2415.162030
8     2338.407661
9     2406.374013
10    2379.051374
Name: posted_rate, dtype: float64

* time/date → rate changes over time


In [186]:
# Execute for analysus only , we shouldn't use in prediction
df["rate_per_mile"] = df["posted_rate"] / df["distance"]

df["rate_per_mile"].describe()

count    48000.000000
mean         2.215347
std          0.583887
min          0.333709
25%          1.986974
50%          2.145348
75%          2.342696
max         14.125294
Name: rate_per_mile, dtype: float64

In [187]:
df.groupby("equipment")["rate_per_mile"].describe()

,count,mean,std,min,25%,50%,75%,max
equipment,,,,,,,,
Dry Van,27202.0,2.115401,0.538102,0.333709,1.920572,2.046203,2.211423,12.816710
Flatbed,8753.0,2.294729,0.610187,0.353881,2.076055,2.219683,2.407007,12.949578
Reefer,12045.0,2.383376,0.615467,0.386292,2.167801,2.311540,2.494937,14.125294


In [188]:
df.nlargest(10, "rate_per_mile")[
    ["load_id", "distance", "posted_rate", "equipment", "weight", "date"]
]

,load_id,distance,posted_rate,equipment,weight,date
22254,TR-022255,314.7,4445.23,Reefer,36485.0,2025-05-21
36609,TR-036610,98.4,1386.42,Reefer,22226.0,2025-08-20
870,TR-000871,305.6,4069.24,Reefer,26447.0,2025-01-06
17070,TR-017071,201.5,2609.34,Flatbed,38714.0,2025-04-18
27969,TR-027970,589.9,7597.52,Reefer,24951.0,2025-06-25
23369,TR-023370,412.4,5308.96,Flatbed,35494.0,2025-05-28
28012,TR-028013,604.3,7773.37,Reefer,39690.0,2025-06-25
45483,TR-045484,153.8,1971.21,Dry Van,34198.0,2025-10-15
1415,TR-001416,198.2,2509.96,Reefer,25383.0,2025-01-09
21968,TR-021969,254.5,3220.66,Dry Van,32993.0,2025-05-19


In [189]:
df.groupby("pickup")["rate_per_mile"].mean().sort_values(ascending=False).head(10)

pickup
Dallas         2.425212
Chattanooga    2.378462
Nashville      2.352082
Atlanta        2.340549
New Orleans    2.338994
Louisville     2.336465
Lexington      2.331034
Tulsa          2.329744
Memphis        2.329540
Mobile         2.323584
Name: rate_per_mile, dtype: float64

In [190]:
df.groupby("delivery")["rate_per_mile"].mean().sort_values(ascending=False).head(10)

delivery
Birmingham     2.364832
Nashville      2.357729
Atlanta        2.353768
Memphis        2.352426
Little Rock    2.345919
Shreveport     2.344782
Dallas         2.341730
Cincinnati     2.332460
Houston        2.330656
Lexington      2.315889
Name: rate_per_mile, dtype: float64

In [191]:
df.groupby(df["date"].dt.month)[
    ["market_index", "quote_signal"]
].mean()

,market_index,quote_signal
date,,
1,0.926696,2.068436
2,0.995997,2.097130
3,1.072880,2.179539
4,1.207649,1.963388
5,1.301452,1.912829
6,1.280316,2.296007
7,1.200893,1.922997
8,0.981480,2.052200
9,0.893039,2.199785


In [192]:
df[df["weight"].isna()]["posted_rate"].describe()

count     300.000000
mean     2350.665767
std      1390.972806
min       148.720000
25%      1229.660000
50%      1984.995000
75%      3397.890000
max      6546.690000
Name: posted_rate, dtype: float64

In [193]:
df[df["market_index"].isna()]["posted_rate"].describe()

count     374.000000
mean     2392.115107
std      1434.640755
min       207.710000
25%      1254.287500
50%      2081.100000
75%      3294.790000
max      9454.940000
Name: posted_rate, dtype: float64

In [194]:
df.groupby(pd.cut(df["distance"], bins=10))["posted_rate"].agg(
    ["count", "mean", "median"]
)

/var/folders/t9/swq9p1cx5hsfc_jq47nbv3zh0000gn/T/ipykernel_95755/3764861283.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(pd.cut(df["distance"], bins=10))["posted_rate"].agg(


,count,mean,median
distance,,,
"(66.63, 406.98]",7119,701.898500,723.280
"(406.98, 743.96]",10612,1315.939616,1290.370
"(743.96, 1080.94]",9635,1970.549415,1934.000
"(1080.94, 1417.92]",5815,2611.280985,2549.600
"(1417.92, 1754.9]",4293,3275.496401,3215.280
"(1754.9, 2091.88]",4359,3896.482709,3825.830
"(2091.88, 2428.86]",3086,4377.450953,4312.945
"(2428.86, 2765.84]",1602,4935.562541,4882.445
"(2765.84, 3102.82]",1204,5549.807209,5449.580


## 2. Feature Engineering


In [195]:
# Build time features before splitting so they exist in both train_df and val_df
df["time_index"] = (df["date"] - df["date"].min()).dt.days
df["month"] = df["date"].dt.month

df[["date", "time_index", "month"]].head()

,date,time_index,month
0,2025-01-01,0,1
1,2025-01-01,0,1
2,2025-01-01,0,1
3,2025-01-01,0,1
4,2025-01-01,0,1


In [196]:
#simple time split
df.groupby(df["date"].dt.to_period("M")).size()

date
2025-01    4918
2025-02    4337
2025-03    5036
2025-04    4819
2025-05    4913
2025-06    4783
2025-07    4912
2025-08    4759
2025-09    4670
2025-10    4853
Freq: M, dtype: int64

In [197]:
df["time_index"] = (df["date"] - df["date"].min()).dt.days
df["month"] = df["date"].dt.month

In [198]:
train_df = df[df["date"] < "2025-10-01"].copy()
val_df = df[df["date"] >= "2025-10-01"].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)

Train: (43147, 17)
Validation: (4853, 17)


## 3.Models 

### 3.1 Model trained by distance

In [199]:
X_train = train_df[["distance"]]
y_train = train_df["posted_rate"]

X_val = val_df[["distance"]]
y_val = val_df["posted_rate"]

baseline = LinearRegression()

baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_val)

baseline_mae = mean_absolute_error(y_val, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_val, baseline_pred))

print("Baseline: Distance only")
print("MAE:", baseline_mae)
print("RMSE:", baseline_rmse)

Baseline: Distance only
MAE: 192.7741904419588
RMSE: 668.2130277041988


## Feature Engineering and Preprocessing

The baseline model uses only distance. Since the dataset also contains information about
weight, market conditions, quote signal, equipment, locations, and time, I will test whether
these features improve the prediction.

For the weight column, negative values were treated as a sign error during data cleaning.
Missing numerical values will be imputed using the median, while categorical variables will
be one-hot encoded.

In [200]:
# Work on copies so the original data remains unchanged
train_model = train_df.copy()
val_model = val_df.copy()

# Correct the detected sign error in weight
train_model["weight"] = train_model["weight"].abs()
val_model["weight"] = val_model["weight"].abs()

### Define numeric and categorical features

In [201]:
numeric_features = [
    "distance",
    "weight",
    "market_index",
    "quote_signal",
    "time_index"
]

categorical_features = [
    "equipment",
    "pickup",
    "delivery"
]

### Preprocessing Pipeline

In [202]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

### Linear Regression with all features

In [203]:
X_train = train_model[numeric_features + categorical_features]
y_train = train_model["posted_rate"]

X_val = val_model[numeric_features + categorical_features]
y_val = val_model["posted_rate"]

linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

linear_model.fit(X_train, y_train)

linear_pred = linear_model.predict(X_val)

linear_mae = mean_absolute_error(y_val, linear_pred)
linear_rmse = np.sqrt(mean_squared_error(y_val, linear_pred))

print("Linear Regression: Full Features")
print("MAE:", linear_mae)
print("RMSE:", linear_rmse)

Linear Regression: Full Features
MAE: 138.35339314384504
RMSE: 651.5676402263398


* Using Full Features have better results rather than using distance only

### Random Forest

In [204]:
random_forest = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

random_forest.fit(X_train, y_train)

rf_pred = random_forest.predict(X_val)

rf_mae = mean_absolute_error(y_val, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_val, rf_pred))

print("Random Forest: Full Features")
print("MAE:", rf_mae)
print("RMSE:", rf_rmse)

Random Forest - Full Features
MAE: 233.42072143004313
RMSE: 748.0359303837545


* Random Forest results are worst than linear regression